# Notebook 04 - Feature Validation

Notebook này đánh giá mức độ phù hợp của các đặc trưng được lựa chọn để xây dựng mô hình dự đoán Digital Burnout.

Quá trình đánh giá bao gồm kiểm định thống kê, đo lường mức độ quan trọng của đặc trưng và xây dựng bảng xếp hạng tổng hợp. Kết quả sẽ được sử dụng làm đầu vào cho Notebook 05 - Modeling.

# 0. Set Up

Nhập các thư viện cần thiết, thiết lập cấu hình chung và khai báo các đường dẫn sử dụng trong notebook.

In [1]:
# Data manipulation

import numpy as np
import pandas as pd

# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis

from scipy.stats import f_oneway
from scipy.stats import chi2_contingency

from sklearn.feature_selection import mutual_info_classif

# Machine learning

from sklearn.ensemble import RandomForestClassifier

# Utilities

from pathlib import Path

import warnings

In [2]:
# Hiển thị toàn bộ cột

pd.set_option(

    "display.max_columns",
    None

)

pd.set_option(

    "display.float_format",
    "{:.4f}".format

)

warnings.filterwarnings(
    "ignore"
)

In [3]:
# Khai báo đường dẫn

project_root = Path.cwd().resolve().parent.parent

input_path = (

    project_root /
    "data" /
    "processed" /
    "international_dataset" /
    "digital_burnout_cleaned.csv"

)

output_directory = (

    project_root /
    "data" /
    "processed" /
    "international_dataset"

)

In [4]:
# Cấu hình chung

RANDOM_STATE = 42

TARGET_VARIABLE = "burnout_level"

MUTUAL_INFORMATION_SAMPLE_SIZE = 500_000

RANDOM_FOREST_SAMPLE_SIZE = 100_000

RANDOM_FOREST_ESTIMATORS = 100

# 1. Load Cleaned Dataset

Đọc bộ dữ liệu đã được tiền xử lý từ Notebook 02 và kiểm tra thông tin cơ bản trước khi thực hiện Feature Validation.

In [5]:
# Đọc bộ dữ liệu

df = pd.read_csv(
    input_path
)

print(
    f"Dataset Shape: {df.shape}"
)

Dataset Shape: (5000000, 35)


In [6]:
# Thông tin bộ dữ liệu

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000000 entries, 0 to 4999999
Data columns (total 35 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   user_id                  int64  
 1   age                      int64  
 2   occupation               object 
 3   work_mode                object 
 4   device_usage_type        object 
 5   daily_screen_time        float64
 6   social_media_hours       float64
 7   doomscrolling_duration   float64
 8   app_switch_frequency     int64  
 9   notification_count       int64  
 10  smartphone_unlocks       int64  
 11  late_night_device_usage  int64  
 12  focus_sessions           int64  
 13  deep_work_hours          float64
 14  distraction_frequency    int64  
 15  task_completion_rate     int64  
 16  concentration_score      int64  
 17  sleep_hours              float64
 18  sleep_quality            int64  
 19  caffeine_intake          int64  
 20  physical_activity        float64
 21  stress_l

In [7]:
# Xem trước dữ liệu

display(
    df.head()
)

,user_id,age,occupation,work_mode,device_usage_type,daily_screen_time,social_media_hours,doomscrolling_duration,app_switch_frequency,notification_count,smartphone_unlocks,late_night_device_usage,focus_sessions,deep_work_hours,distraction_frequency,task_completion_rate,concentration_score,sleep_hours,sleep_quality,caffeine_intake,physical_activity,stress_level,workspace_quality,meeting_hours,internet_stability,remote_work_days,motivation_level,mental_fatigue,emotional_exhaustion,work_satisfaction,mental_state,burnout_score,productivity_score,productivity_category,burnout_level
0,1,56,Content Creator,Office,Entertainment-Centric,8.8000,5.0000,1.2000,41,112,49,1,3,6.0000,67,92,2,5.9000,10,6,1.6000,10,5,2.8000,3,4,8.0000,10,4,8,Balanced,46,100,High,Moderate
1,2,46,Student,Hybrid,Work-Centric,10.3000,2.2000,2.4000,119,168,153,1,9,4.6000,75,74,3,5.6000,7,5,1.1000,5,10,3.2000,7,6,5.0000,7,9,7,Balanced,57,96,High,Moderate
2,3,32,Software Engineer,Remote,Balanced,6.5000,4.6000,1.0000,121,199,234,1,5,3.0000,70,96,7,5.5000,8,6,1.1000,4,1,2.3000,9,6,8.0000,5,2,6,Balanced,29,79,High,Low
3,4,25,Designer,Office,Balanced,9.6000,1.2000,0.1000,85,122,177,1,9,2.9000,107,60,3,6.1000,5,1,1.7000,1,4,3.8000,10,5,6.0000,4,5,3,Burnout,57,63,Medium,Moderate
4,5,38,Analyst,Hybrid,Work-Centric,13.3000,1.6000,1.9000,221,73,91,1,9,2.8000,72,73,9,7.9000,1,3,1.8000,4,8,3.0000,1,1,4.0000,7,9,7,Focused,64,89,High,Moderate


In [8]:
# Kiểm tra giá trị thiếu

missing_summary = (

    df
    .isnull()
    .sum()
    .to_frame(
        name="Missing Values"
    )

)

display(
    missing_summary
)

,Missing Values
user_id,0
age,0
occupation,0
work_mode,0
device_usage_type,0
daily_screen_time,0
social_media_hours,0
doomscrolling_duration,0
app_switch_frequency,0
notification_count,0


In [9]:
# Xây dựng Burnout Risk từ Burnout Score

df["burnout_risk"] = pd.cut(

    df["burnout_score"],
    bins=[0, 4, 7, 10],
    
    labels=[

        "Low",
        "Moderate",
        "High"

    ],
    include_lowest=True
)

print(
    df["burnout_risk"].value_counts()
)

burnout_risk
Low         117717
High         58752
Moderate     45826
Name: count, dtype: int64


In [10]:
# Tổng hợp thông tin bộ dữ liệu

dataset_summary = pd.DataFrame({

    "Metric": [

        "Number of Records",
        "Number of Features",
        "Target Variable"

    ],

    "Value": [

        len(df),
        df.shape[1],
        TARGET_VARIABLE

    ]

})

display(
    dataset_summary
)

,Metric,Value
0,Number of Records,5000000
1,Number of Features,36
2,Target Variable,burnout_level


# 2. Feature Validation Configuration

Định nghĩa tập đặc trưng sử dụng trong quá trình Feature Validation, bao gồm các Digital Burnout Indicators và Context Features phục vụ xây dựng mô hình dự đoán.

### 2.1 Digital Burnout Indicators

Định nghĩa các Digital Burnout Indicators được xây dựng từ khung lý thuyết.

In [11]:
# Digital Burnout Indicators

dbi_features = {

    "Burnout Symptoms": [

        "emotional_exhaustion",
        "stress_level",
        "mental_fatigue"

    ],

    "Digital Exposure": [

        "daily_screen_time",
        "social_media_hours",
        "doomscrolling_duration",
        "late_night_device_usage",
        "notification_count",
        "smartphone_unlocks",
        "app_switch_frequency"

    ],

    "Learning and Productivity": [

        "concentration_score",
        "distraction_frequency",
        "focus_sessions",
        "deep_work_hours",
        "task_completion_rate"

    ],

    "Sleep and Recovery": [

        "sleep_hours",
        "sleep_quality",
        "motivation_level"

    ]

}

### 2.2 Context Features

Định nghĩa các biến ngữ cảnh được bổ sung từ bộ dữ liệu quốc tế nhằm hỗ trợ mô hình Machine Learning.

In [12]:
# Context Features

context_features = [

    "work_mode",
    "device_usage_type"

]

### 2.3 Modeling Features

Kết hợp Digital Burnout Indicators và Context Features thành tập đặc trưng đầu vào cho quá trình Feature Validation.

In [13]:
# Tổng hợp Modeling Features

dbi_feature_list = [

    feature
    for feature_group in dbi_features.values()
    for feature in feature_group

]

modeling_features = (

    dbi_feature_list +
    context_features

)

print(
    f"Tổng số đặc trưng mô hình: {len(modeling_features)}"
)

Tổng số đặc trưng mô hình: 20


### 2.4 Numerical and Categorical Features

Phân chia tập đặc trưng theo kiểu dữ liệu để phục vụ các phương pháp kiểm định thống kê và Machine Learning.

In [14]:
# Numerical Features

numerical_features = dbi_feature_list

# Categorical Features

categorical_features = context_features

print(
    f"Số đặc trưng số: {len(numerical_features)}"
)

print(
    f"Số đặc trưng phân loại: {len(categorical_features)}"
)

Số đặc trưng số: 18
Số đặc trưng phân loại: 2


### 2.5 Validation Summary

Tóm tắt cấu hình các đặc trưng được sử dụng trong quá trình Feature Validation.

In [15]:
# Tổng hợp cấu hình Feature Validation

feature_configuration = pd.DataFrame({

    "Thông tin": [

        "Số Digital Burnout Indicators",
        "Số Context Features",
        "Tổng số đặc trưng mô hình",
        "Biến mục tiêu"

    ],

    "Giá trị": [

        len(dbi_feature_list),
        len(context_features),
        len(modeling_features),
        TARGET_VARIABLE

    ]

})

display(
    feature_configuration
)

,Thông tin,Giá trị
0,Số Digital Burnout Indicators,18
1,Số Context Features,2
2,Tổng số đặc trưng mô hình,20
3,Biến mục tiêu,burnout_level


# 3. Statistical Feature Validation

Đánh giá mối quan hệ giữa từng đặc trưng và biến mục tiêu bằng các phương pháp kiểm định thống kê phù hợp với từng kiểu dữ liệu.

### 3.1 ANOVA F-test

Đánh giá khả năng phân biệt giữa các nhóm Burnout Risk của các đặc trưng dạng số bằng kiểm định ANOVA một chiều.

In [16]:
# Tính ANOVA F-test

anova_results = []

for feature in numerical_features:

    burnout_groups = [

        group[feature].values
        for _, group in df.groupby(TARGET_VARIABLE)

    ]

    f_statistic, p_value = f_oneway(
        *burnout_groups
    )

    anova_results.append({

        "Feature": feature,
        "F Statistic": f_statistic,
        "P Value": p_value

    })

anova_results = pd.DataFrame(
    anova_results
)

display(
    anova_results
)

,Feature,F Statistic,P Value
0,emotional_exhaustion,693837.6903,0.0000
1,stress_level,339866.5449,0.0000
2,mental_fatigue,0.0056,0.9944
3,daily_screen_time,101564.9368,0.0000
4,social_media_hours,0.0259,0.9744
5,doomscrolling_duration,83030.4357,0.0000
6,late_night_device_usage,51096.7093,0.0000
7,notification_count,18957.1395,0.0000
8,smartphone_unlocks,0.7454,0.4746
9,app_switch_frequency,1.0597,0.3466


In [17]:
# Xếp hạng ANOVA

anova_results = (

    anova_results

    .sort_values(

        by="F Statistic",
        ascending=False

    )

    .reset_index(
        drop=True
    )

)
anova_results["ANOVA Rank"] = (
    anova_results.index + 1
)

display(
    anova_results
)

,Feature,F Statistic,P Value,ANOVA Rank
0,emotional_exhaustion,693837.6903,0.0000,1
1,stress_level,339866.5449,0.0000,2
2,daily_screen_time,101564.9368,0.0000,3
3,doomscrolling_duration,83030.4357,0.0000,4
4,late_night_device_usage,51096.7093,0.0000,5
5,distraction_frequency,47473.5884,0.0000,6
6,sleep_hours,33841.7953,0.0000,7
7,notification_count,18957.1395,0.0000,8
8,deep_work_hours,4322.0935,0.0000,9
9,focus_sessions,3.6071,0.0271,10


In [18]:
# Tổng hợp kết quả ANOVA

anova_summary = pd.DataFrame({

    "Chỉ số": [

        "Tổng số đặc trưng",
        "Đặc trưng có ý nghĩa thống kê",
        "Đặc trưng không có ý nghĩa thống kê"

    ],

    "Giá trị": [

        len(anova_results),
        (anova_results["P Value"] < 0.05).sum(),
        (anova_results["P Value"] >= 0.05).sum()

    ]

})

display(anova_summary)

,Chỉ số,Giá trị
0,Tổng số đặc trưng,18
1,Đặc trưng có ý nghĩa thống kê,10
2,Đặc trưng không có ý nghĩa thống kê,8


### 3.2 Chi-square Test

Đánh giá mối liên hệ giữa các đặc trưng phân loại và biến mục tiêu bằng kiểm định Chi-square.

In [19]:
# Tính Chi-square Test

chi_square_results = []

for feature in categorical_features:

    contingency_table = pd.crosstab(

        df[feature],
        df[TARGET_VARIABLE]

    )

    chi2_statistic, p_value, _, _ = chi2_contingency(
        contingency_table
    )

    chi_square_results.append({

        "Feature": feature,
        "Chi-square Statistic": chi2_statistic,
        "P Value": p_value

    })

chi_square_results = pd.DataFrame(
    chi_square_results
)

display(
    chi_square_results
)

,Feature,Chi-square Statistic,P Value
0,work_mode,1.3624,0.8507
1,device_usage_type,1.3614,0.8509


In [20]:
# Xếp hạng Chi-square

chi_square_results = (

    chi_square_results

    .sort_values(

        by="Chi-square Statistic",
        ascending=False

    )

    .reset_index(
        drop=True
    )

)

chi_square_results["Chi-square Rank"] = (
    chi_square_results.index + 1
)

display(
    chi_square_results
)

,Feature,Chi-square Statistic,P Value,Chi-square Rank
0,work_mode,1.3624,0.8507,1
1,device_usage_type,1.3614,0.8509,2


In [21]:
# Tổng hợp kết quả Chi-square

chi_square_summary = pd.DataFrame({

    "Chỉ số": [

        "Tổng số đặc trưng",
        "Đặc trưng có ý nghĩa thống kê",
        "Đặc trưng không có ý nghĩa thống kê"

    ],

    "Giá trị": [

        len(chi_square_results),
        (chi_square_results["P Value"] < 0.05).sum(),
        (chi_square_results["P Value"] >= 0.05).sum()

    ]

})

display(
    chi_square_summary
)

,Chỉ số,Giá trị
0,Tổng số đặc trưng,2
1,Đặc trưng có ý nghĩa thống kê,0
2,Đặc trưng không có ý nghĩa thống kê,2


**Nhận xét**

Kết quả kiểm định Chi-square cho thấy các Context Features không có mối liên hệ có ý nghĩa thống kê với Burnout Risk trên bộ dữ liệu hiện tại. Tuy nhiên, các đặc trưng này vẫn được giữ lại để tiếp tục đánh giá bằng Mutual Information và Random Forest Feature Importance, đồng thời phục vụ xây dựng mô hình dự đoán ở Notebook 05.

### 3.3 Mutual Information

Đánh giá mức độ phụ thuộc giữa từng đặc trưng và Burnout Risk bằng Mutual Information nhằm phát hiện cả các mối quan hệ phi tuyến.

In [22]:
# Lấy mẫu dữ liệu

sampled_df = df.sample(

    n=MUTUAL_INFORMATION_SAMPLE_SIZE,
    random_state=RANDOM_STATE

).copy()

print(
    f"Kích thước mẫu: {len(sampled_df):,}"
)

Kích thước mẫu: 500,000


In [23]:
# Chuẩn bị dữ liệu

feature_dataset = sampled_df[
    modeling_features
].copy()

target_dataset = sampled_df[
    TARGET_VARIABLE
].copy()

In [24]:
# Mã hóa biến phân loại

for feature in categorical_features:

    feature_dataset[feature] = (

        feature_dataset[feature]
        .astype("category")
        .cat.codes

    )

target_dataset = (

    target_dataset
    .astype("category")
    .cat.codes

)

In [25]:
# Xác định biến phân loại

discrete_features = [

    feature in categorical_features
    for feature in feature_dataset.columns

]

# Tính Mutual Information

mi_scores = mutual_info_classif(

    X=feature_dataset,
    y=target_dataset,
    discrete_features=discrete_features,
    random_state=RANDOM_STATE

)

mutual_information_results = pd.DataFrame({

    "Feature": feature_dataset.columns,
    "MI Score": mi_scores

})

display(
    mutual_information_results

)

,Feature,MI Score
0,emotional_exhaustion,0.1257
1,stress_level,0.0676
2,mental_fatigue,0.0041
3,daily_screen_time,0.0195
4,social_media_hours,0.0010
5,doomscrolling_duration,0.0156
6,late_night_device_usage,0.0514
7,notification_count,0.0043
8,smartphone_unlocks,0.0003
9,app_switch_frequency,0.0002


In [26]:
# Xếp hạng Mutual Information

mutual_information_results = (

    mutual_information_results

    .sort_values(

        by="MI Score",
        ascending=False

    )

    .reset_index(
        drop=True
    )

)

mutual_information_results["MI Rank"] = (
    mutual_information_results.index + 1
)

In [27]:
# Tổng hợp Mutual Information

mutual_information_summary = pd.DataFrame({

    "Chỉ số": [

        "Số đặc trưng",
        "Điểm MI lớn nhất",
        "Điểm MI trung bình",
        "Điểm MI nhỏ nhất"

    ],

    "Giá trị": [

        len(mutual_information_results),
        mutual_information_results["MI Score"].max(),
        mutual_information_results["MI Score"].mean(),
        mutual_information_results["MI Score"].min()

    ]

})

display(
    mutual_information_summary
)

,Chỉ số,Giá trị
0,Số đặc trưng,20.0000
1,Điểm MI lớn nhất,0.1257
2,Điểm MI trung bình,0.0163
3,Điểm MI nhỏ nhất,0.0000


# 4. Feature Importance Assessment

Đánh giá mức độ quan trọng của từng đặc trưng bằng Random Forest và tổng hợp kết quả từ các phương pháp đánh giá để xây dựng bảng xếp hạng cuối cùng.

### 4.1 Random Forest Feature Importance

Đánh giá mức độ đóng góp của từng đặc trưng đối với việc dự đoán Burnout Risk bằng Random Forest.

In [28]:
# Lấy mẫu dữ liệu

training_sample = df.sample(

    n=100_000,
    random_state=RANDOM_STATE

)

print(
    f"Kích thước mẫu huấn luyện: {len(training_sample):,}"
)

Kích thước mẫu huấn luyện: 100,000


In [29]:
# Chuẩn bị dữ liệu

training_dataset = training_sample[
    modeling_features
].copy()

training_target = training_sample[
    TARGET_VARIABLE
].copy()

In [30]:
# Mã hóa biến phân loại

for feature in categorical_features:

    training_dataset[feature] = (

        training_dataset[feature]
        .astype("category")
        .cat.codes

    )

training_target = (

    training_target
    .astype("category")
    .cat.codes

)

In [31]:
# Huấn luyện Random Forest

random_forest_model = RandomForestClassifier(

    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1

)

random_forest_model.fit(

    training_dataset,
    training_target

)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [32]:
# Tính Feature Importance

random_forest_results = pd.DataFrame({

    "Feature": training_dataset.columns,
    "RF Score": random_forest_model.feature_importances_

})

random_forest_results = (

    random_forest_results

    .sort_values(

        by="RF Score",
        ascending=False

    )

    .reset_index(
        drop=True
    )

)

random_forest_results["RF Rank"] = (
    random_forest_results.index + 1
)

display(
    random_forest_results
)

,Feature,RF Score,RF Rank
0,emotional_exhaustion,0.1248,1
1,stress_level,0.0828,2
2,daily_screen_time,0.0733,3
3,distraction_frequency,0.0638,4
4,notification_count,0.0612,5
5,doomscrolling_duration,0.0609,6
6,sleep_hours,0.0567,7
7,smartphone_unlocks,0.0555,8
8,app_switch_frequency,0.0553,9
9,deep_work_hours,0.0526,10


In [33]:
# Tổng hợp Feature Importance

random_forest_summary = pd.DataFrame({

    "Chỉ số": [

        "Số đặc trưng",
        "Điểm Importance lớn nhất",
        "Điểm Importance trung bình",
        "Điểm Importance nhỏ nhất"

    ],

    "Giá trị": [

        len(random_forest_results),
        random_forest_results["RF Score"].max(),
        random_forest_results["RF Score"].mean(),
        random_forest_results["RF Score"].min()

    ]

})

display(
    random_forest_summary
)

,Chỉ số,Giá trị
0,Số đặc trưng,20.0000
1,Điểm Importance lớn nhất,0.1248
2,Điểm Importance trung bình,0.0500
3,Điểm Importance nhỏ nhất,0.0123


### 4.2 Feature Rank Aggregation

Tổng hợp thứ hạng của các đặc trưng từ nhiều phương pháp đánh giá nhằm xây dựng bảng xếp hạng tổng thể.

In [34]:
# Khởi tạo bảng tổng hợp

feature_ranking = pd.DataFrame({
    "Feature": modeling_features
})

In [35]:
# Ghép ANOVA Rank

feature_ranking = feature_ranking.merge(

    anova_results[
        ["Feature", "ANOVA Rank"]
    ],
    on="Feature",
    how="left"

)

In [36]:
# Ghép Mutual Information Rank

feature_ranking = feature_ranking.merge(

    mutual_information_results[
        ["Feature", "MI Rank"]
    ],
    on="Feature",
    how="left"

)

In [37]:
# Ghép Random Forest Rank

feature_ranking = feature_ranking.merge(

    random_forest_results[
        ["Feature", "RF Rank"]
    ],
    on="Feature",
    how="left"

)

#### Calculate Overall Rank

Tính thứ hạng trung bình của từng đặc trưng.

In [38]:
# Tính Overall Rank

feature_ranking["Aggregated Rank"] = (

    feature_ranking[

        [

            "ANOVA Rank",
            "MI Rank",
            "RF Rank"

        ]

    ]

    .mean(

        axis=1,
        skipna=True

    )

)

In [39]:
feature_ranking = (

    feature_ranking

    .sort_values(
        by="Aggregated Rank"
    )

    .reset_index(
        drop=True
    )

)

### 4.3 Overall Feature Ranking

Xếp hạng cuối cùng của các đặc trưng dựa trên kết quả tổng hợp từ nhiều phương pháp đánh giá.

In [40]:
# Xếp hạng cuối cùng

feature_ranking["Final Rank"] = (
    feature_ranking.index + 1
)

display(
    feature_ranking
)

,Feature,ANOVA Rank,MI Rank,RF Rank,Aggregated Rank,Final Rank
0,emotional_exhaustion,1.0000,1,1,1.0000,1
1,stress_level,2.0000,2,2,2.0000,2
2,daily_screen_time,3.0000,4,3,3.3333,3
3,doomscrolling_duration,4.0000,5,6,5.0000,4
4,distraction_frequency,6.0000,6,4,5.3333,5
5,sleep_hours,7.0000,7,7,7.0000,6
6,notification_count,8.0000,10,5,7.6667,7
7,late_night_device_usage,5.0000,3,20,9.3333,8
8,deep_work_hours,9.0000,14,10,11.0000,9
9,focus_sessions,10.0000,8,17,11.6667,10


In [41]:
# Top 20 đặc trưng

feature_ranking

,Feature,ANOVA Rank,MI Rank,RF Rank,Aggregated Rank,Final Rank
0,emotional_exhaustion,1.0000,1,1,1.0000,1
1,stress_level,2.0000,2,2,2.0000,2
2,daily_screen_time,3.0000,4,3,3.3333,3
3,doomscrolling_duration,4.0000,5,6,5.0000,4
4,distraction_frequency,6.0000,6,4,5.3333,5
5,sleep_hours,7.0000,7,7,7.0000,6
6,notification_count,8.0000,10,5,7.6667,7
7,late_night_device_usage,5.0000,3,20,9.3333,8
8,deep_work_hours,9.0000,14,10,11.0000,9
9,focus_sessions,10.0000,8,17,11.6667,10


**Note**

Aggregated Rank được tính từ các phương pháp đánh giá phù hợp với từng loại đặc trưng. Đối với các biến phân loại (Context Features), ANOVA không được áp dụng nên Average Rank được tính từ Mutual Information và Random Forest.

## 5. Validated Feature Summary

Phần này tổng hợp kết quả của quá trình đánh giá đặc trưng nhằm xác định mức độ ưu tiên của các đặc trưng được sử dụng trong mô hình dự đoán Digital Burnout.

Dựa trên kết quả từ ANOVA F-test, Mutual Information và Random Forest Feature Importance, các đặc trưng được xếp hạng theo phương pháp Rank Aggregation và phân nhóm thành Primary, Secondary và Supporting Features. Kết quả này là cơ sở để xây dựng bộ đặc trưng đầu vào cho Notebook 05 - Modeling.

### 5.1 Feature Classification

Phân loại các đặc trưng theo mức độ ưu tiên dựa trên kết quả xếp hạng tổng hợp.

In [42]:
# Phân loại đặc trưng theo thứ hạng

def classify_feature(rank):

    if rank <= 7:
        return "Primary Feature"
    elif rank <= 14:
        return "Secondary Feature"
    else:
        return "Supporting Feature"


feature_ranking["Feature Category"] = (

    feature_ranking["Final Rank"]
    .apply(classify_feature)

)

display(
    feature_ranking
)

,Feature,ANOVA Rank,MI Rank,RF Rank,Aggregated Rank,Final Rank,Feature Category
0,emotional_exhaustion,1.0000,1,1,1.0000,1,Primary Feature
1,stress_level,2.0000,2,2,2.0000,2,Primary Feature
2,daily_screen_time,3.0000,4,3,3.3333,3,Primary Feature
3,doomscrolling_duration,4.0000,5,6,5.0000,4,Primary Feature
4,distraction_frequency,6.0000,6,4,5.3333,5,Primary Feature
5,sleep_hours,7.0000,7,7,7.0000,6,Primary Feature
6,notification_count,8.0000,10,5,7.6667,7,Primary Feature
7,late_night_device_usage,5.0000,3,20,9.3333,8,Secondary Feature
8,deep_work_hours,9.0000,14,10,11.0000,9,Secondary Feature
9,focus_sessions,10.0000,8,17,11.6667,10,Secondary Feature


In [43]:
# Thống kê số lượng đặc trưng theo nhóm

feature_category_summary = (

    feature_ranking["Feature Category"]
    .value_counts()
    .rename_axis(
        "Feature Category"
    )

    .reset_index(
        name="Count"
    )

)

display(
    feature_category_summary
)

,Feature Category,Count
0,Primary Feature,7
1,Secondary Feature,7
2,Supporting Feature,6


### 5.2 Validation Summary

Tóm tắt kết quả đánh giá đặc trưng sau khi tổng hợp từ nhiều phương pháp thống kê và học máy. Phần này cung cấp cái nhìn tổng quan về số lượng đặc trưng theo từng nhóm ưu tiên và xác nhận bộ đặc trưng được sử dụng cho giai đoạn xây dựng mô hình.

In [44]:
# Thống kê kết quả đánh giá đặc trưng

validation_summary = pd.DataFrame(

    {

        "Metric": [

            "Total Research Features",
            "Primary Features",
            "Secondary Features",
            "Supporting Features",
            "Modeling Features"

        ],

        "Value": [
            len(feature_ranking),
            (

                feature_ranking["Feature Category"]
                == "Primary Feature"

            ).sum(),
            (

                feature_ranking["Feature Category"]
                == "Secondary Feature"

            ).sum(),

            (

                feature_ranking["Feature Category"]
                == "Supporting Feature"

            ).sum(),
            len(feature_ranking)

        ]

    }

)

display(
    validation_summary
)

,Metric,Value
0,Total Research Features,20
1,Primary Features,7
2,Secondary Features,7
3,Supporting Features,6
4,Modeling Features,20


In [45]:
print(
    f"Tổng số đặc trưng nghiên cứu: {len(feature_ranking)}"
)

print(
    f"Primary Features: {(feature_ranking['Feature Category'] == 'Primary Feature').sum()}"
)

print(
    f"Secondary Features: {(feature_ranking['Feature Category'] == 'Secondary Feature').sum()}"
)

print(
    f"Supporting Features: {(feature_ranking['Feature Category'] == 'Supporting Feature').sum()}"
)

print(
    "Tất cả các đặc trưng sẽ được sử dụng trong Notebook 05 - Modeling."
)

Tổng số đặc trưng nghiên cứu: 20
Primary Features: 7
Secondary Features: 7
Supporting Features: 6
Tất cả các đặc trưng sẽ được sử dụng trong Notebook 05 - Modeling.


### 5.3 Final Validated Features

Tổng hợp danh sách đặc trưng cuối cùng sau quá trình đánh giá và xếp hạng. Bộ đặc trưng này sẽ được sử dụng làm đầu vào cho giai đoạn xây dựng mô hình trong Notebook 05.

In [46]:
# Danh sách đặc trưng cuối cùng

validated_features = (

    feature_ranking[

        [

            "Final Rank",
            "Feature",
            "Feature Category"

        ]

    ]

    .sort_values(
        by="Final Rank"
    )

    .reset_index(
        drop=True
    )

)

display(
    validated_features
)

,Final Rank,Feature,Feature Category
0,1,emotional_exhaustion,Primary Feature
1,2,stress_level,Primary Feature
2,3,daily_screen_time,Primary Feature
3,4,doomscrolling_duration,Primary Feature
4,5,distraction_frequency,Primary Feature
5,6,sleep_hours,Primary Feature
6,7,notification_count,Primary Feature
7,8,late_night_device_usage,Secondary Feature
8,9,deep_work_hours,Secondary Feature
9,10,focus_sessions,Secondary Feature


In [47]:
# Danh sách đặc trưng sử dụng cho mô hình

modeling_features = (

    validated_features["Feature"]
    .tolist()

)

print(
    "Danh sách đặc trưng cho mô hình:"
)

print(
    modeling_features
)

Danh sách đặc trưng cho mô hình:
['emotional_exhaustion', 'stress_level', 'daily_screen_time', 'doomscrolling_duration', 'distraction_frequency', 'sleep_hours', 'notification_count', 'late_night_device_usage', 'deep_work_hours', 'focus_sessions', 'smartphone_unlocks', 'motivation_level', 'sleep_quality', 'app_switch_frequency', 'concentration_score', 'task_completion_rate', 'social_media_hours', 'mental_fatigue', 'work_mode', 'device_usage_type']


In [48]:
print(
    f"Tổng số đặc trưng được xác nhận: {len(modeling_features)}"
)

print(
    "Bộ đặc trưng đã sẵn sàng cho Notebook 05 - Modeling."
)

Tổng số đặc trưng được xác nhận: 20
Bộ đặc trưng đã sẵn sàng cho Notebook 05 - Modeling.


## 6. Research Findings

Kết quả đánh giá đặc trưng cho thấy bộ 20 đặc trưng được lựa chọn từ cơ sở lý thuyết đều có giá trị trong việc mô tả Digital Burnout ở các mức độ khác nhau.

Các phương pháp đánh giá bao gồm ANOVA F-test, Chi-square Test, Mutual Information và Random Forest Feature Importance đã cung cấp các góc nhìn bổ sung về mức độ liên quan và đóng góp của từng đặc trưng đối với biến mục tiêu. Kết quả được tổng hợp thông qua phương pháp Rank Aggregation nhằm xây dựng bảng xếp hạng đặc trưng cuối cùng.

Nhóm Primary Features bao gồm các đặc trưng có mức độ ưu tiên cao và đóng vai trò nổi bật trong việc dự đoán Digital Burnout. Các Secondary Features tiếp tục bổ sung thông tin hữu ích cho mô hình, trong khi Supporting Features mặc dù có mức độ ưu tiên thấp hơn nhưng vẫn được giữ lại nhằm khai thác các mối quan hệ bổ sung và tương tác giữa các đặc trưng trong quá trình học của mô hình.

Bộ đặc trưng cuối cùng sẽ được sử dụng làm đầu vào cho Notebook 05 để xây dựng và đánh giá các mô hình dự đoán Digital Burnout.

## 7. Export Results

Xuất toàn bộ kết quả đánh giá đặc trưng để sử dụng trong các giai đoạn tiếp theo của nghiên cứu.

In [49]:
# Thư mục lưu kết quả

output_directory = (

    project_root
    / "data"
    / "processed"
    / "international_dataset"

)

output_directory.mkdir(

    parents=True,
    exist_ok=True

)

print(
    f"Thư mục lưu kết quả: {output_directory}"
)

Thư mục lưu kết quả: D:\DA016\data\processed\international_dataset


In [50]:
# Xuất danh sách đặc trưng cuối cùng

validated_features.to_csv(

    output_directory
    / "validated_features.csv",
    index=False

)

print(
    "Đã xuất validated_features.csv"
)

Đã xuất validated_features.csv


In [51]:
# Xuất kết quả đánh giá đặc trưng

feature_validation_results = (
    feature_ranking.copy()
)

feature_validation_results.to_csv(

    output_directory
    / "feature_validation_results.csv",
    index=False

)

print(
    "Đã xuất feature_validation_results.csv"
)

Đã xuất feature_validation_results.csv


In [52]:
# Xuất bảng tổng hợp

validation_summary.to_csv(

    output_directory
    / "feature_validation_summary.csv",
    index=False

)

print(
    "Đã xuất feature_validation_summary.csv"
)

Đã xuất feature_validation_summary.csv
